# Lesson 18: Securing AI Agents with Cryptographic Receipts

## Hands-on Notebook

This notebook walks through four exercises:

1. **Sign your first receipt** for an agent tool call and verify it.
2. **Tamper with the receipt** and watch verification fail.
3. **Build a three-receipt chain** and confirm chain integrity.
4. **Wrap a Microsoft Agent Framework tool call** so every action emits a receipt.

All cryptographic primitives are imported from well-maintained libraries (`pynacl` for Ed25519, `jcs` for RFC 8785 canonical JSON, `hashlib` from the Python standard library for SHA-256). The receipt logic itself is plain Python that you can read and modify.

Run the cells in order. Each section is short and self-contained.

## Setup

Install the two dependencies. Both have permissive licenses (Apache-2.0 / MIT).

In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Helper Utilities

These two helpers handle base64url encoding (without padding) and SHA-256 hashing of arbitrary objects. They keep the rest of the notebook focused on the receipt logic itself.

In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1: Sign your first receipt

Imagine our agent for **Contoso Travel** just looked up flights from Sydney to Los Angeles for a customer. We want to record this tool call as a signed receipt so a future auditor can verify it without trusting us.

### Step 1.1: Generate a signing key

In production, the agent's signing key would live in a hardware security module (HSM), Azure Key Vault, or a similar protected store. For this lesson we generate a fresh key in memory.

In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Step 1.2: Build the receipt payload

The payload contains everything we want the receipt to attest to: who acted, what tool, with what arguments, what came back, under what policy, and when. We hash the arguments and result rather than including them inline so the receipt does not leak sensitive content.

In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Step 1.3: Sign and assemble the receipt

Three steps:

1. Canonicalize the payload using JCS so two implementations producing the same logical receipt produce byte-identical bytes.
2. Hash the canonical bytes with SHA-256.
3. Sign the hash with the Ed25519 private key.

The signature is then attached to the original payload to produce the final receipt.

In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Step 1.4: Verify the receipt

Verification reverses the process. We strip the signature, recompute the canonical hash, and check the signature against the public key in the receipt.

An auditor doing this verification needs nothing from us except the receipt itself. No service to call, no key directory to query, no trust required.

In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

You should see `Receipt is valid: True`. The agent has produced its first cryptographically signed audit record.

## Section 2: Tamper with the receipt

The whole point of receipts is that they are tamper-evident. Let's prove it.

We will modify exactly one character of the receipt and watch verification fail.

In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### What just happened?

When we changed `policy_id`, the canonical bytes changed. The SHA-256 hash of those bytes changed. The signature (which was over the original hash) no longer matches the new hash. Verification correctly returns `False`.

There is no way to modify any field of the receipt and still have it verify, unless the attacker has the private key. As long as the private key is in a key vault and the public key is published, tampering is impossible to hide.

Try it yourself: modify the `tool_name` or `agent_id` or `timestamp` in the cell above and re-run. Every change produces an invalid receipt.

## Section 3: Chain receipts together

A single receipt protects one action. Most agents take many actions. To make the entire sequence tamper-evident, we link each receipt to the previous one by including the previous receipt's hash in the new receipt's payload.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

If anyone removes or reorders a receipt, the chain breaks at exactly that point. Verification of any later receipt fails because its `previous_receipt_hash` no longer matches the actual hash of its predecessor.

In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Now break the chain by tampering with the middle receipt and re-verify. The tampered receipt fails its signature check, AND the next receipt fails its chain-link check (because its `previous_receipt_hash` no longer matches the modified middle receipt's hash).

In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 still verifies (it was not modified and has no predecessor to depend on). Receipt 1 fails its signature check because we changed `tool_args_hash`. Receipt 2 fails its chain-link check because its `previous_receipt_hash` was computed against the original (now-modified) receipt 1.

Even if an attacker re-signs the modified receipt 1 (which they cannot do without the private key), the chain-link mismatch in receipt 2 would still expose the tampering. To hide the change, the attacker would have to re-sign every receipt from the modification point onward, which requires possession of the private key.

## Section 4: Wrap an agent tool call with receipt signing

In a real deployment, you do not want every agent author to remember to call `make_receipt`. You want receipt signing to be automatic for every tool invocation.

Here is the simplest pattern: a wrapper class that takes any callable tool function and returns a receipt-emitting version of it. This adapts to any agent framework, including the Microsoft Agent Framework (`agent_framework.azure`).

If you do not have an Azure AI Foundry project set up, the local mock below still demonstrates the pattern.

In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Integrating with the Microsoft Agent Framework

The `ReceiptedTool` wrapper above is framework-agnostic. To use it inside an agent built with the Microsoft Agent Framework, register the wrapped function as a tool. A sketch (you would replace the mock with a real Azure AI Foundry tool registration):

```python
# Pseudocode showing the integration shape.
# from agent_framework.azure import AzureAIProjectAgentProvider
# from azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

The agent framework does not need to know anything about receipts. Receipt signing is wrapped around the tool, not bolted into the framework. This is how you add provenance to existing agent code without rewriting the agent.

## Recap and stretch challenge

You have:

- Generated an Ed25519 key pair.
- Built and signed a receipt for an agent tool call.
- Verified the receipt offline using only the public key.
- Tampered with a receipt and observed verification fail.
- Built a hash-chained sequence of three receipts.
- Tampered with the middle of the chain and observed both signature failure and chain-link failure.
- Wrapped a tool function with automatic receipt signing.

**Stretch challenge.** Extend the receipt schema with a `request_id` field (a UUID for distributed tracing). Update `make_receipt` to include it, and confirm receipts still verify end to end. Then modify the field after signing and confirm verification fails. This forces you to internalize how every byte of the canonical encoding contributes to the signature.

**Important boundary.** Receipts prove three things and only three things: attribution (this key signed this content), integrity (the content has not changed since signing), and ordering (this receipt came after that receipt). They do NOT prove that the agent's action was correct, that the policy named in `policy_id` was actually evaluated, or that the agent followed every rule. Receipts are a foundation. Governance is the system you build on top.

Read the lesson README again with that boundary in mind. The most common mistake teams make with receipts is assuming "we have receipts" means "we are governed." It does not. Receipts make agent behavior auditable. They do not make it correct.